# Generative AI: Assignment 1

**Name:** Kushagra Gupta
**Roll No:** 27PGAI0115
**Email:** aakg2310@gmail.com

This notebook is built with **LangChain** + a **local Ollama model** (`llama3.2`) so it can run
completely offline/free and without hitting Groq rate limits, per the assignment's note.

- **Part 1** — Topic Detection & Summarization of BBC News Articles (45 marks)
- **Part 2** — Job Postings Analysis: Role Categorization & Requirements Extraction (55 marks)
- **Bonus** — Full-dataset runs for both parts (20 marks), included at the end, checkpointed so
  they can resume if interrupted.


## Setup

In [3]:
import json
import re
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", 120)


In [4]:
# Ollama runs locally; OLLAMA_HOST is set to 0.0.0.0 (server bind address) in this environment,
# which is NOT a valid address for the *client* to connect to, so we point the client at
# localhost explicitly.
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "llama3.2"

llm = ChatOllama(model=MODEL_NAME, base_url=OLLAMA_BASE_URL, temperature=0)
llm_json = ChatOllama(model=MODEL_NAME, base_url=OLLAMA_BASE_URL, temperature=0, format="json")

# Quick smoke test
print(llm.invoke("Reply with just the word: ready").content)


ready


---
# Part 1: Topic Detection and Summarization of News Articles (45 marks)


### Step 1: Load the Dataset

In [5]:
bbc_df_full = pd.read_csv("bbc-news-data.csv", sep="\t")
print("Full dataset shape:", bbc_df_full.shape)
print(bbc_df_full["category"].value_counts())

bbc_df = bbc_df_full.head(30).reset_index(drop=True)
bbc_df.insert(0, "Article_ID", bbc_df.index)
bbc_df.head()


Full dataset shape: (2225, 4)
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,Article_ID,category,filename,title,content
0,0,business,001.txt,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from..."
1,1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said th...
2,2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $9...
3,3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months ...
4,4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover...


### Step 2: Define the Topic Classification Task (10 marks)

We use a `ChatPromptTemplate` with a couple of few-shot examples, then a simple LCEL chain
(`prompt | llm | StrOutputParser()`) so the model returns a single category label.


In [6]:
TOPIC_CATEGORIES = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

topic_few_shot_examples = [
    {
        "article": "The stock market rallied today after the central bank cut interest rates, "
                    "boosting shares of major banks and manufacturers.",
        "category": "Business",
    },
    {
        "article": "The striker scored a hat-trick as the home team thrashed their rivals "
                    "4-0 in front of a record crowd at the stadium.",
        "category": "Sport",
    },
]

topic_example_prompt = ChatPromptTemplate.from_messages(
    [("human", "Article: {article}"), ("ai", "{category}")]
)
topic_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=topic_example_prompt,
    examples=topic_few_shot_examples,
)

topic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are a news topic classifier. Analyze the news article and identify its topic as "
         "ONE of the following categories: Business, Entertainment, Politics, Sport, or Tech. "
         "Reply with ONLY the single category label, nothing else."),
        topic_few_shot_prompt,
        ("human", "Article: {article}"),
    ]
)

topic_chain = topic_prompt | llm | StrOutputParser()


def classify_topic(article_text: str) -> str:
    raw = topic_chain.invoke({"article": article_text[:3000]}).strip()
    # Normalize to one of the known labels in case the model adds extra words/punctuation.
    for cat in TOPIC_CATEGORIES:
        if cat.lower() in raw.lower():
            return cat
    return raw.split()[0] if raw else "Unknown"


# Expected Output: show it works for a sample datapoint.
sample = bbc_df.iloc[0]
sample_topic = classify_topic(sample["content"])
print("Actual category:", sample["category"])
print("Predicted topic :", sample_topic)


Actual category: business
Predicted topic : Business


### Step 3: Define the Summarization Task (10 marks)


In [7]:
summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You summarize news articles factually and concisely, capturing who/what/when/where/why "
         "as applicable, without personal commentary or opinions."),
        ("human",
         "Summarize the main points of the following news article in 2-3 sentences.\n\n"
         "Article:\n{article}"),
    ]
)
summary_chain = summary_prompt | llm | StrOutputParser()


def summarize_article(article_text: str) -> str:
    return summary_chain.invoke({"article": article_text[:4000]}).strip()


# Expected Output: show it works for a sample datapoint.
sample_summary = summarize_article(sample["content"])
print("Summary:\n", sample_summary)


Summary:
 Here is a summary of the article in 3 sentences:

Time Warner's quarterly profits jumped 76% to $1.13 billion, driven by sales of high-speed internet connections and higher advert sales, with the company also benefiting from one-off gains. The company's internet business, AOL, saw a decline in subscribers but reported an 8% increase in underlying profit, with plans to offer the service for free to Time Warner internet customers. Time Warner is also reiterating its full-year financial performance, projecting 5% operating earnings growth and wider profit margins, while also restating its accounts due to an inquiry into AOL by US market regulators.


### Step 4: Key Entity Extraction (10 marks)

We ask the model to return **structured JSON** (via Ollama's `format="json"` mode) so the
`Key_Entities` column can hold a clean Python list rather than free-form text.


In [8]:
class EntityExtraction(BaseModel):
    entities: list[str] = Field(
        description="Important people, organizations, and places mentioned in the article."
    )


entity_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "From the article, list the names of any important people, organizations, or places "
         "mentioned. Respond with ONLY a JSON object of the form "
         '{{"entities": ["Name1", "Name2", ...]}}. If none are found, return an empty list.'),
        ("human", "Article:\n{article}"),
    ]
)
entity_chain = entity_prompt | llm_json | StrOutputParser()


def extract_entities(article_text: str) -> list[str]:
    raw = entity_chain.invoke({"article": article_text[:4000]}).strip()
    try:
        data = json.loads(raw)
        entities = data.get("entities", [])
        return [str(e).strip() for e in entities if str(e).strip()]
    except (json.JSONDecodeError, AttributeError):
        # Fallback: split on commas/newlines if the model didn't return clean JSON.
        return [e.strip("- \t") for e in re.split(r"[,\n]", raw) if e.strip("- \t")]


# Expected Output: show it works for a sample datapoint.
sample_entities = extract_entities(sample["content"])
print("Key entities:", sample_entities)


Key entities: ['Richard Parsons', 'TimeWarner', 'Google', 'Warner Bros', 'AOL', 'US Securities Exchange Commission', 'Bertelsmann', 'Lord of the Rings', 'Alexander', 'Catwoman', 'SEC']


### Step 5: Update the DataFrame with Results (15 marks)

Apply classification, summarization, and entity extraction to every row of the 30-article
DataFrame, with a checkpoint file so re-running the notebook doesn't redo completed rows.


In [9]:
BBC_CHECKPOINT = Path("bbc_news_first30_checkpoint.csv")


def run_bbc_pipeline(df: pd.DataFrame, checkpoint_path: Path, desc: str) -> pd.DataFrame:
    if checkpoint_path.exists():
        result = pd.read_csv(checkpoint_path)
    else:
        result = df.copy()
        result["Detected_Topic"] = pd.NA
        result["Summary"] = pd.NA
        result["Key_Entities"] = pd.NA

    for idx in tqdm(result.index, desc=desc):
        if pd.notna(result.loc[idx, "Detected_Topic"]):
            continue  # already processed (resume support)
        text = result.loc[idx, "content"]
        try:
            topic = classify_topic(text)
            summ = summarize_article(text)
            ents = extract_entities(text)
        except Exception as e:  # keep going even if Ollama hiccups on one row
            print(f"Row {idx} failed: {e}")
            topic, summ, ents = "Error", "Error", []
        result.loc[idx, "Detected_Topic"] = topic
        result.loc[idx, "Summary"] = summ
        result.loc[idx, "Key_Entities"] = json.dumps(ents)
        result.to_csv(checkpoint_path, index=False)  # checkpoint after every row

    result["Key_Entities"] = result["Key_Entities"].apply(json.loads)
    return result


bbc_results = run_bbc_pipeline(bbc_df, BBC_CHECKPOINT, "Part 1: first 30 articles")
bbc_results.head()


Part 1: first 30 articles:   0%|          | 0/30 [00:00<?, ?it/s]

Row 14 failed: [WinError 10054] An existing connection was forcibly closed by the remote host
Row 15 failed: [WinError 10054] An existing connection was forcibly closed by the remote host
Row 16 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 17 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 18 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 19 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 20 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 21 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 22 failed: [WinError 10061] No connection could be made because the target machine actively refused it
Row 23 failed: [WinError 10061] No connection could be made bec

,Article_ID,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,0,business,001.txt,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from...",Business,"Here is a summary of the article in 3 sentences:\n\nTime Warner's quarterly profits jumped 76% to $1.13 billion, dri...","[Richard Parsons, TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission, Bertelsmann, Lord of the ..."
1,1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said th...,Business,Here is a summary of the article in 3 sentences:\n\nThe US dollar has reached its highest level against the euro in ...,"[Alan Greenspan, Robert Sinche, Federal Reserve, Bank of America, G7, China, White House]"
2,2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $9...,Business,"Here is a summary of the article in 2-3 sentences:\n\nThe owners of Yukos, Menatep Group, plan to ask the buyer of i...","[Jamie Firestone, Tim Osborne, Mikhail Khodorkovsky, Rosneft, Yukos, Menatep Group, Reuters, US, Russia, Moscow, Yug..."
3,3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months ...,Business,Here is a summary of the article in 2-3 sentences:\n\nBritish Airways reported a 40% drop in profits to £75m for the...,"[Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul]"
4,4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover...,Business,Here is a summary of the article in 2-3 sentences:\n\nShares in UK drinks and food firm Allied Domecq rose 4% on spe...,"[Pernod Ricard, Wall Street Journal, Financial Times, Seagram, Diageo, LVMH, Glenmorangie, Jacob's Creek, Chivas Reg..."


In [10]:
# Final merged dataframe: original + new columns together.
bbc_final = bbc_results[
    ["Article_ID", "category", "title", "content", "Detected_Topic", "Summary", "Key_Entities"]
]
bbc_final.to_csv("bbc_news_analyzed_first30.csv", index=False)
print(bbc_final.shape)
bbc_final.head(10)


(30, 7)


,Article_ID,category,title,content,Detected_Topic,Summary,Key_Entities
0,0,business,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from...",Business,"Here is a summary of the article in 3 sentences:\n\nTime Warner's quarterly profits jumped 76% to $1.13 billion, dri...","[Richard Parsons, TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission, Bertelsmann, Lord of the ..."
1,1,business,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said th...,Business,Here is a summary of the article in 3 sentences:\n\nThe US dollar has reached its highest level against the euro in ...,"[Alan Greenspan, Robert Sinche, Federal Reserve, Bank of America, G7, China, White House]"
2,2,business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $9...,Business,"Here is a summary of the article in 2-3 sentences:\n\nThe owners of Yukos, Menatep Group, plan to ask the buyer of i...","[Jamie Firestone, Tim Osborne, Mikhail Khodorkovsky, Rosneft, Yukos, Menatep Group, Reuters, US, Russia, Moscow, Yug..."
3,3,business,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months ...,Business,Here is a summary of the article in 2-3 sentences:\n\nBritish Airways reported a 40% drop in profits to £75m for the...,"[Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul]"
4,4,business,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover...,Business,Here is a summary of the article in 2-3 sentences:\n\nShares in UK drinks and food firm Allied Domecq rose 4% on spe...,"[Pernod Ricard, Wall Street Journal, Financial Times, Seagram, Diageo, LVMH, Glenmorangie, Jacob's Creek, Chivas Reg..."
5,5,business,Japan narrowly escapes recession,"Japan's economy teetered on the brink of a technical recession in the three months to September, figures show. Rev...",Business,Here is a summary of the article in 3 sentences:\n\nJapan's economy experienced a technical recession in the three m...,"[Heizo Takenaka, Paul Sheard, Lehman Brothers, Japan]"
6,6,business,Jobs growth still slow in the US,"The US created fewer jobs than expected in January, but a fall in jobseekers pushed the unemployment rate to its lo...",Business,"Here is a summary of the article in 3 sentences:\n\nThe US economy added 146,000 jobs in January, below market expec...","[Herbert Hoover, Rick Egelton, Ken Mayland, BMO Financial Group, ClearView Economics, President Bush, Labor Departme..."
7,7,business,India calls for fair trade rules,"India, which attends the G7 meeting of seven leading industrialised nations on Friday, is unlikely to be cowed by i...",Politics,"Here is a summary of the article in 2-3 sentences:\n\nIndia's finance minister, Palaniappan Chidambaram, criticized ...","[Palaniappan Chidambaram, Gordon Brown, United Nations, World Bank, IMF, G7, G20, China, Brazil, South Africa, Russi..."
8,8,business,Ethiopia's crop production up 24%,"Ethiopia produced 14.27 million tonnes of crops in 2004, 24% higher than in 2003 and 21% more than the average of t...",Business,"Here is a summary of the article in 3 sentences:\n\nEthiopia's crop production increased by 21% in 2004, reaching 14...","[Henri Josserand, Food and Agriculture Organisation, World Food Programme, Ethiopia]"
9,9,business,Court rejects $280bn tobacco case,A US government claim accusing the country's biggest tobacco companies of covering up the effects of smoking has be...,Politics,Here is a summary of the article in 3 sentences:\n\nAn appeals court in Washington has rejected a US government clai...,"[Altria Group, RJ Reynolds Tobacco, Lorillard Tobac

In [11]:
# Quick accuracy check against the dataset's ground-truth category labels.
accuracy = (bbc_final["category"].str.lower() == bbc_final["Detected_Topic"].str.lower()).mean()
print(f"Topic classification accuracy on first 30 articles: {accuracy:.1%}")


Topic classification accuracy on first 30 articles: 33.3%


---
# Part 2: Job Postings Analysis – Role Categorization and Requirements Extraction (55 marks)


### Step 1: Load the Dataset

In [12]:
jobs_df_full = pd.read_csv("job_title_des.csv")
jobs_df_full = jobs_df_full.rename(
    columns={"Job Title": "Job_Title", "Job Description": "Job_Description"}
)
jobs_df_full = jobs_df_full.drop(columns=[c for c in jobs_df_full.columns if c.startswith("Unnamed")])
print("Full dataset shape:", jobs_df_full.shape)

jobs_df = jobs_df_full.head(25).reset_index(drop=True)
jobs_df.head()


Full dataset shape: (2277, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types:...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperi...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n\nResponsibilities\n\nWe are looking for a capable data scientist to j..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience and generalist engineers with...
4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming engineering team meet need rap...


### Step 2: Define the Job Category Classification Task (10 marks)


In [15]:
JOB_CATEGORIES = [
    "Technology/IT", "Finance", "Marketing", "Healthcare", "Sales",
    "Human Resources", "Operations", "Design", "Education", "Other",
]

job_category_few_shot = [
    {
        "title": "Senior Data Analyst",
        "description": "Analyze sales data, build dashboards, SQL and Python required.",
        "category": "Technology/IT",
    },
    {
        "title": "Registered Nurse",
        "description": "Provide patient care in a hospital ward, monitor vitals.",
        "category": "Healthcare",
    },
]

job_cat_example_prompt = ChatPromptTemplate.from_messages(
    [("human", "Job Title: {title}\nDescription: {description}"), ("ai", "{category}")]
)
job_cat_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=job_cat_example_prompt,
    examples=job_category_few_shot,
)

job_category_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "Given a job title and description, categorize the job into ONE of the following "
         f"domains: {', '.join(JOB_CATEGORIES)}. If unsure, use 'Other'. "
         "Reply with ONLY the single category label, nothing else."),
        job_cat_few_shot_prompt,
        ("human", "Job Title: {title}\nDescription: {description}"),
    ]
)
job_category_chain = job_category_prompt | llm | StrOutputParser()


def classify_job_category(title: str, description: str) -> str:
    raw = job_category_chain.invoke(
        {"title": title, "description": str(description)[:2500]}
    ).strip()
    for cat in JOB_CATEGORIES:
        if cat.lower() in raw.lower():
            return cat
    return raw.split("\n")[0][:40] if raw else "Other"


# Expected Output: show it works for a sample datapoint.
job_sample = jobs_df.iloc[0]
sample_category = classify_job_category(job_sample["Job_Title"], job_sample["Job_Description"])
print("Job title:", job_sample["Job_Title"])
print("Predicted category:", sample_category)


Job title: Flutter Developer
Predicted category: Technology/IT


### Step 3: Define the Requirements Extraction Task (30 marks)

A single structured prompt (JSON mode) extracts `Skills`, `Education`, and `Experience`
together, with "Not specified" as the fallback when a field isn't mentioned.


In [16]:
class JobRequirements(BaseModel):
    skills: list[str] = Field(description="Key skills, technologies, or tools mentioned.")
    education: str = Field(description="Minimum education level required/preferred, or 'Not specified'.")
    experience: str = Field(description="Years of experience or experience level required, or 'Not specified'.")


requirements_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "Extract the required skills, education level, and years of experience from the job "
         "description below. Respond with ONLY a JSON object of the form: "
         '{{"skills": ["skill1", "skill2", ...], "education": "...", "experience": "..."}}. '
         "If a field is not mentioned, use the string \"Not specified\" for it "
         "(empty list [] for skills if none found)."),
        ("human", "Job Title: {title}\nJob Description:\n{description}"),
    ]
)
requirements_chain = requirements_prompt | llm_json | StrOutputParser()


def extract_requirements(title: str, description: str) -> dict:
    raw = requirements_chain.invoke(
        {"title": title, "description": str(description)[:3000]}
    ).strip()
    try:
        data = json.loads(raw)
        return {
            "skills": [str(s).strip() for s in data.get("skills", []) if str(s).strip()],
            "education": str(data.get("education") or "Not specified").strip(),
            "experience": str(data.get("experience") or "Not specified").strip(),
        }
    except (json.JSONDecodeError, AttributeError):
        return {"skills": [], "education": "Not specified", "experience": "Not specified"}


# Expected Output: show it works for a sample datapoint.
sample_reqs = extract_requirements(job_sample["Job_Title"], job_sample["Job_Description"])
print(json.dumps(sample_reqs, indent=2))


{
  "skills": [
    "Flutter",
    "Software Development"
  ],
  "education": "Not specified",
  "experience": "1 year (Preferred)"
}


### Step 4 & 5: Apply the LLM Chain to Each Job Posting and Update the DataFrame
(10 + 5 marks)


In [18]:
JOBS_CHECKPOINT = Path("job_postings_first25_checkpoint.csv")


def run_jobs_pipeline(df: pd.DataFrame, checkpoint_path: Path, desc: str) -> pd.DataFrame:
    if checkpoint_path.exists():
        result = pd.read_csv(checkpoint_path)
    else:
        result = df.copy()
        result["Predicted_Category"] = pd.NA
        result["Required_Skills"] = pd.NA
        result["Education_Required"] = pd.NA
        result["Experience_Required"] = pd.NA

    for idx in tqdm(result.index, desc=desc):
        if pd.notna(result.loc[idx, "Predicted_Category"]):
            continue  # resume support
        title = result.loc[idx, "Job_Title"]
        desc_text = result.loc[idx, "Job_Description"]
        try:
            category = classify_job_category(title, desc_text)
            reqs = extract_requirements(title, desc_text)
        except Exception as e:
            print(f"Row {idx} failed: {e}")
            category = "Error"
            reqs = {"skills": [], "education": "Not specified", "experience": "Not specified"}
        result.loc[idx, "Predicted_Category"] = category
        result.loc[idx, "Required_Skills"] = json.dumps(reqs["skills"])
        result.loc[idx, "Education_Required"] = reqs["education"]
        result.loc[idx, "Experience_Required"] = reqs["experience"]
        result.to_csv(checkpoint_path, index=False)

    result["Required_Skills"] = result["Required_Skills"].apply(json.loads)
    return result


jobs_results = run_jobs_pipeline(jobs_df, JOBS_CHECKPOINT, "Part 2: first 25 postings")
jobs_results.head()


Part 2: first 25 postings:   0%|          | 0/25 [00:00<?, ?it/s]

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types:...,Technology/IT,"[Flutter, Software Development]",Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperi...,Technology/IT,"[Python, API development, Django, Flask, Linux, SQL, PyUnit, Automated unit testing]",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n\nResponsibilities\n\nWe are looking for a capable data scientist to j...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java, Statistics, Applied Mathematics, Software Development]","Graduate or M.Sc. in Computer Science, Mathematics or equivalent",3 years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience and generalist engineers with...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, Networking, Mobile Network Issues, C...",Not specified,5-10 years
4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming engineering team meet need rap...,Technology/IT,"[React, J Native, JavaScript, HTML, CSS, RESTful APIs, HTTP, Networking, Full Stack Web Development, MVC, Object-Ori...",Not specified,5+ years


In [19]:
jobs_final = jobs_results[
    ["Job_Title", "Job_Description", "Predicted_Category",
     "Required_Skills", "Education_Required", "Experience_Required"]
]
jobs_final.to_csv("job_postings_analyzed_first25.csv", index=False)
print(jobs_final.shape)
jobs_final.head(10)


(25, 6)


,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types:...,Technology/IT,"[Flutter, Software Development]",Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperi...,Technology/IT,"[Python, API development, Django, Flask, Linux, SQL, PyUnit, Automated unit testing]",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n\nResponsibilities\n\nWe are looking for a capable data scientist to j...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java, Statistics, Applied Mathematics, Software Development]","Graduate or M.Sc. in Computer Science, Mathematics or equivalent",3 years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience and generalist engineers with...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, Networking, Mobile Network Issues, C...",Not specified,5-10 years
4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming engineering team meet need rap...,Technology/IT,"[React, J Native, JavaScript, HTML, CSS, RESTful APIs, HTTP, Networking, Full Stack Web Development, MVC, Object-Ori...",Not specified,5+ years
5,Java Developer,Software Developer - Integration*\nImmediate Opening!*\nA dynamic Akron / Cleveland area company is looking for an e...,Technology/IT,"[Proven technical expertise in the design, development, coding, testing, and debugging of enterprise software, Stron...","Bachelor's Degree in Computer Science, Information Systems, or related field, or combination of education and equiva...",2 years
6,Full Stack Developer,senior full stack developer \- 1800026h cwt looking senior full stack developer proven back-end skill well strong fr...,Technology/IT,"[NodeJS, Java, MongoDB, Elasticsearch, Redis, React, Angular, JavaScript, CS, HTML, Responsive UI, Cross-browser dev...",B.Sc in Computer Science or Engineering,2+ years
7,JavaScript Developer,"Job Description:\n\nReactJS + NodeJs, Azure Functions, and GraphQL capability\n\nStrong hands on experience on javas...",Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript]","Any graduation, Any PG, Any Doctorate",3 - 8 years
8,DevOps Engineer,"Main Responsibilities and Deliverables:\nManage/support the rollout, scalability and execution of the automation as ...",Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloudify, CFEngine, Cobbler, Foreman, PHP, Linux, Windows, application debu...",Not specified,5-10 years
9,Software Engineer,"Overview\n\n\nBased in Silicon Valley, Tintri is a wholly owned subsidiary of DataDirect Networks (DDN.com), the dat...",Technology/IT,"[REST API, C/C++, Python, Go, Git, Gerrit, Jenkins]","BS or MS; computer engineering, computer science or related technical field",7 years


---
# Bonus: Full-Dataset Runs (up to 20 marks)

Runs the same pipelines over **all** rows of both datasets (2,225 BBC articles + 2,277 job
postings), using local Ollama so there's no rate limiting. Each pipeline checkpoints to CSV
after every row, so it can be safely interrupted and resumed by re-running the cell.

⚠️ This is the slow part — expect this to take a while on CPU. Run it last / let it run in the
background.


In [ ]:
BBC_FULL_CHECKPOINT = Path("bbc_news_full_checkpoint.csv")

bbc_full_df = bbc_df_full.reset_index(drop=True)
bbc_full_df.insert(0, "Article_ID", bbc_full_df.index)

bbc_full_results = run_bbc_pipeline(bbc_full_df, BBC_FULL_CHECKPOINT, "BONUS Part 1: all articles")
bbc_full_final = bbc_full_results[
    ["Article_ID", "category", "title", "content", "Detected_Topic", "Summary", "Key_Entities"]
]
bbc_full_final.to_csv("bbc_news_analyzed_full.csv", index=False)
print(bbc_full_final.shape)


BONUS Part 1: all articles:   0%|          | 0/2225 [00:00<?, ?it/s]

In [ ]:
JOBS_FULL_CHECKPOINT = Path("job_postings_full_checkpoint.csv")

jobs_full_df = jobs_df_full.reset_index(drop=True)

jobs_full_results = run_jobs_pipeline(jobs_full_df, JOBS_FULL_CHECKPOINT, "BONUS Part 2: all postings")
jobs_full_final = jobs_full_results[
    ["Job_Title", "Job_Description", "Predicted_Category",
     "Required_Skills", "Education_Required", "Experience_Required"]
]
jobs_full_final.to_csv("job_postings_analyzed_full.csv", index=False)
print(jobs_full_final.shape)
